# DuckPD Feature Workflows

This notebook demonstrates advanced DuckPD workflows on **real-world financial market data** using the [AlphaDojo/dojo_stock_news](https://huggingface.co/datasets/AlphaDojo/dojo_stock_news) dataset (~3.9M articles).

### What you will see:
- **Direct Remote Parquet Scanning**: Query millions of rows in cloud parquet without loading full datasets into Python memory.
- **Vectorized String Accessors (`.str`)**: Clean publisher names, extract headlines, and filter topics lazily.
- **Multi-Table Relational Merges (`merge`)**: Join multi-million row news feeds with ticker metadata tables.
- **Multi-Frame Concatenation (`duckpd.concat`)**: Combine filtered partitions with automatic schema union and null-padding.
- **Extended Reductions**: Compute standard deviation (`std`), variance (`var`), median (`median`), and quantiles (`quantile`).
- **Advanced Multi-Column GroupBy**: Named aggregations across publishers and tickers.
- **Window & Positional Transforms**: Cumulative, rank, difference, rolling, expanding, and shifted analytics with guaranteed ordering.
- **Persistence, Query Plans & Direct Parquet Export**: Reuse materialized intermediates, inspect pushdown, and write without pandas fallback.

## 1. Setup Session & Connect to Remote Parquet

Initialize a DuckPD session with custom memory and execution settings, then lazily scan the 3.9M row dataset hosted on Hugging Face.

In [ ]:
from pathlib import Path

import pandas as std_pd

import duckpd as pd

print(f"DuckPD Version: {pd.__version__}")
session = pd.connect(memory_limit="1GB", threads=4)

# Remote dataset from AlphaDojo (~3.9M financial news rows)
DATA_URL = "https://huggingface.co/datasets/AlphaDojo/dojo_stock_news/resolve/main/data.parquet"

# Lazily scan Parquet over HTTP; DuckPD preserves its physical file order automatically.
news_df = session.read_parquet(DATA_URL)

print("Lazy DataFrame created:")
print(f"Columns: {news_df.columns}")
print(f"Session executions so far: {session.execution_count}")

## 2. Vectorized String Accessors (`.str`)

Clean publisher names, compute headline lengths, and flag earnings-related announcements lazily using DuckPD's `.str` accessor methods.

In [ ]:
# Perform lazy string feature engineering
enriched_news = news_df.assign(
    publisher_clean=news_df["publisher"].str.strip().str.upper(),
    title_len=news_df["title"].str.len(),
    is_earnings=news_df["title"].str.upper().str.contains("EARNINGS"),
    is_option_activity=news_df["title"].str.contains("Option Activity"),
)

# Inspect a bounded preview pushed down to DuckDB
preview_cols = ["symbol", "publisher_clean", "title_len", "is_earnings", "title"]
enriched_news[preview_cols].head(5)

## 3. Multi-Table Relational Merging (`merge`)

Join the multi-million row news dataset with a ticker reference metadata table. The join and predicates are compiled into relational SQL execution.

In [ ]:
# Reference table for prominent tech & consumer market cap leaders
ticker_meta = session.from_pandas(
    std_pd.DataFrame(
        {
            "symbol": ["AAPL", "NVDA", "MSFT", "AMZN", "TSLA", "GOOGL"],
            "company_name": [
                "Apple Inc.",
                "NVIDIA Corp.",
                "Microsoft Corp.",
                "Amazon.com Inc.",
                "Tesla Inc.",
                "Alphabet Inc.",
            ],
            "sector": [
                "Technology",
                "Semiconductors",
                "Software",
                "E-Commerce",
                "Automotive",
                "Communication",
            ],
            "market_tier": [
                "Mega Cap",
                "Mega Cap",
                "Mega Cap",
                "Mega Cap",
                "Mega Cap",
                "Mega Cap",
            ],
        }
    )
)

# Merge ticker metadata with news stream and sort by publish_date
news_with_sector = ticker_meta.merge(enriched_news, on="symbol", how="inner").sort_values(
    "publish_date"
)

news_with_sector[
    ["symbol", "company_name", "sector", "publisher_clean", "title_len", "title"]
].head(5)

## 4. Multi-Frame Concatenation (`duckpd.concat`)

Combine distinct ticker news subsets row-wise with automatic schema union and null-padding.

In [ ]:
# Split subsets and enrich one partition with custom category tags
nvda_news = news_with_sector[news_with_sector["symbol"] == "NVDA"].assign(focus_area="AI Hardware")[
    ["symbol", "company_name", "focus_area", "publisher_clean", "title"]
]

tsla_news = news_with_sector[news_with_sector["symbol"] == "TSLA"][
    ["symbol", "company_name", "publisher_clean", "title"]
]

# Concatenate partitions: focus_area will be padded with NULLs for TSLA
combined_stream = pd.concat([nvda_news, tsla_news])
print("Union Columns:", combined_stream.columns)

combined_stream.head(6)

## 5. Extended Statistical & Boolean Reductions

Calculate statistical metrics across headline length and content properties (`mean`, `median`, `std`, `var`, `quantile`, `any`, `all`) computed in a single SQL query in DuckDB.

In [ ]:
print("--- Headline Length Statistical Metrics ---")
print(f"Mean Length:       {news_with_sector['title_len'].mean():.2f}")
print(f"Median Length:     {news_with_sector['title_len'].median():.2f}")
print(f"Std Deviation:     {news_with_sector['title_len'].std():.2f}")
print(f"Variance:          {news_with_sector['title_len'].var():.2f}")
print(f"25th Percentile:   {news_with_sector['title_len'].quantile(0.25):.2f}")
print(f"75th Percentile:   {news_with_sector['title_len'].quantile(0.75):.2f}")
print(f"95th Percentile:   {news_with_sector['title_len'].quantile(0.95):.2f}")

print("\n--- Boolean Reductions on Filtered Subset ---")
print(f"All headlines mention earnings? {news_with_sector['is_earnings'].all()}")
print(f"Any headline mentions earnings? {news_with_sector['is_earnings'].any()}")

## 6. Advanced GroupBy & Multi-Metric Aggregations

Perform analytical grouping across publishers and tickers using named aggregations, calculating article volume, average length, dispersion, and extreme values.

In [ ]:
# Aggregate news analytics by publisher across top market-cap tickers
publisher_analytics = (
    news_with_sector.groupby(["publisher_clean"], as_index=False)
    .agg(
        article_count=("title", "count"),
        avg_headline_len=("title_len", "mean"),
        std_headline_len=("title_len", "std"),
        max_headline_len=("title_len", "max"),
        min_headline_len=("title_len", "min"),
    )
    .sort_values("article_count", ascending=False)
)

publisher_analytics.head(10)

## 7. Window & Positional Transforms (`cumsum`, `rank`, `diff`)

Execute analytical window operations over ordered streams. CSV, Parquet, pandas, and Arrow sources carry an automatic ordering guarantee; SQL/table relations and post-join workflows must establish one with `order_by` or `sort_values`.

In [ ]:
# Compute cumulative article counts, volume ranks, and incremental step differences
ranked_publishers = publisher_analytics.assign(
    volume_rank=publisher_analytics["article_count"].rank(method="dense", ascending=False),
    cumulative_articles=publisher_analytics["article_count"].cumsum(),
    article_step_diff=publisher_analytics["article_count"].diff(-1),
)

ranked_publishers.head(10)

## 8. Rolling, Expanding & Shifted Analytics

Build richer ordered analytics with row-based rolling and expanding windows. The transforms stay lazy and compile into DuckDB window expressions; persisting then creates a reusable DuckDB table at an explicit execution boundary.

In [ ]:
publisher_windows = ranked_publishers.assign(
    rolling_3_avg_articles=ranked_publishers["article_count"].rolling(3, min_periods=1).mean(),
    expanding_articles=ranked_publishers["article_count"].expanding().sum(),
    previous_publisher_articles=ranked_publishers["article_count"].shift(1),
)

persisted_publishers = publisher_windows.persist("publisher_window_summary")
print(f"Executions after persist: {session.execution_count}")
persisted_publishers.head(10)

## 9. Plan Inspection (`explain()`) & Direct Export

Inspect the relational query plan generated by DuckPD, including predicate pushdown, joins, and window functions. Then write the window-enriched summary directly to Parquet without routing the full result through pandas.

In [ ]:
print("=== Compiled Query Plan with Window Transforms ===")
print(publisher_windows.explain())

# Keep generated output outside the source tree.
demo_directory = Path("demo") if Path("demo").is_dir() else Path("..")
artifact_directory = demo_directory / ".artifacts"
artifact_directory.mkdir(exist_ok=True)
summary_output = artifact_directory / "stock_news_summary.parquet"
publisher_windows.write_parquet(summary_output, overwrite=True)
print(f"\nExported {summary_output} directly via DuckDB!")